# 01: Load Preprocessed Model Inputs

This notebook demonstrates how to load preprocessed EGM signals and labels, inspect dataset structure, apply lightweight preprocessing, and prepare train/validation/test splits for modeling.

## Workflow:
1. Set up the environment and load configuration
2. Generate mock inputs or load preprocessed arrays from disk
3. Inspect array shapes, class balance, and sample statistics
4. Visualize representative EGM waveforms
5. Apply filtering and normalization used before training
6. Create stratified train/validation/test splits

In [ ]:
# Setup: Change to project root if running from notebook directory
import os
import sys

# Add src to path
project_root = os.path.abspath("..")  
sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Import project modules
from src import config, data_loading, data_processing, utils

print("Imports successful!")

In [ ]:
# Load configuration
cfg = config.get_config("config.yaml")
cfg.ensure_directories_exist()

print("Configuration loaded:")
print(f"  Data directory: {cfg.paths['data_dir']}")
print(f"  Output directory: {cfg.paths['output_dir']}")
print(f"  EGM parameters:")
print(f"    - Sampling rate: {cfg.get_data_param('egm_sampling_rate')} Hz")
print(f"    - Duration: {cfg.get_data_param('egm_duration')} seconds") 
print(f"    - Channels: {cfg.get_data_param('egm_n_channels')}")
print(f"    - Samples: {cfg.get_data_param('egm_n_samples')}")

## Option 1: Generate Mock Model Inputs

In [ ]:
# Generate synthetic data for demo
print("Generating synthetic EGM data...")

n_samples = 50  # Small dataset for quick testing
n_channels = 3
n_timesteps = 2500

# Generate signals and labels
egm_signals = data_processing.generate_synthetic_egm_data(
    n_samples=n_samples,
    n_channels=n_channels,
    n_timesteps=n_timesteps,
    random_state=42
)

labels = data_processing.generate_synthetic_labels(
    n_samples=n_samples,
    n_classes=3,
    class_balance=0.25,
    random_state=42
)

print(f"\nGenerated data shapes:")
print(f"  Signals: {egm_signals.shape}")
print(f"  Labels: {labels.shape}")

## Option 2: Load Preprocessed Arrays From Disk

In [ ]:
# Uncomment to load preprocessed arrays instead of generating mock data
# try:
#     signals_dict, labels_dict = data_loading.load_egm_data(cfg.paths['data_dir'])
#     egm_signals = signals_dict['signals']
#     labels = labels_dict['labels']
#     n_samples = egm_signals.shape[0]
#     print("Loaded preprocessed arrays:")
#     print(f"  Signals: {egm_signals.shape}")
#     print(f"  Labels: {labels.shape}")
# except FileNotFoundError as error:
#     print(f"Preprocessed inputs not found: {error}")
#     print("Falling back to generated mock data")

## Explore Data Structure

In [ ]:
# Print data information
utils.print_batch_info(egm_signals, labels, "Full Dataset")

# Display statistics
print("\nLabel Distribution:")
label_names = cfg.get_data_param('label_names')
label_counts = np.sum(labels, axis=0)
for i, name in enumerate(label_names):
    count = label_counts[i]
    pct = 100 * count / n_samples
    print(f"  {name}: {count:3d} samples ({pct:5.1f}%)")

# Count samples by label type
n_scar = np.sum(np.sum(labels, axis=1) > 0)
n_no_scar = n_samples - n_scar
print(f"  With scar: {n_scar} samples ({100*n_scar/n_samples:.1f}%)")
print(f"  No scar: {n_no_scar} samples ({100*n_no_scar/n_samples:.1f}%)")

## Visualize Sample Signals

In [ ]:
# Plot sample signals from each class
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f"Channel {i}" for i in range(3)] + 
                   [f"Channel {i} (with scar)" for i in range(3)] +
                   [f"Channel {i}" for i in range(3)]
)

# Find examples
idx_no_scar = np.where(np.sum(labels, axis=1) == 0)[0][0]  
idx_any_scar = np.where(np.sum(labels, axis=1) > 0)[0][0]
idx_multi_scar = np.where(np.sum(labels, axis=1) > 1)[0][0] if np.any(np.sum(labels, axis=1) > 1) else idx_any_scar

t = np.linspace(0, 2.5, 2500)
samples = [idx_no_scar, idx_any_scar, idx_multi_scar]

for col, sample_idx in enumerate(samples):
    for row in range(3):  # 3 channels
        signal = egm_signals[sample_idx, row, :]
        fig.add_trace(
            go.Scatter(x=t, y=signal, mode='lines', name=f"Sample {sample_idx}",
                      showlegend=(row==0)),
            row=row+1, col=col+1
        )

fig.update_xaxes(title_text="Time (s)")
fig.update_yaxes(title_text="Amplitude (mV)")
fig.update_layout(height=700, title_text="Sample EGM Signals")
fig.show()

## Data Preprocessing: Filtering and Normalization

In [ ]:
# Apply bandpass filter
print("Applying bandpass filter (1-250 Hz)...")
egm_filtered = data_processing.bandpass_filter(
    egm_signals,
    lowcut=1.0,
    highcut=250.0,
    sampling_rate=1000.0,
    order=4
)

print(f"Filtered signal shape: {egm_filtered.shape}")
print(f"Filtered signal range: [{egm_filtered.min():.4f}, {egm_filtered.max():.4f}]")

In [ ]:
# Normalize channels
print("Normalizing channels to peak amplitude...")
egm_normalized = data_processing.normalize_channels(egm_filtered, method="peak")

print(f"Normalized signal shape: {egm_normalized.shape}")
print(f"Normalized signal range: [{egm_normalized.min():.4f}, {egm_normalized.max():.4f}]")
print(f"Max abs value per channel: {np.max(np.abs(egm_normalized), axis=(0, 2))}")

## Stratified Train/Val/Test Split

In [ ]:
# Split data
print("Splitting data into train/val/test sets...")

(X_train, y_train), (X_val, y_val), (X_test, y_test) = data_processing.split_data_stratified(
    egm_normalized,
    labels,
    test_ratio=0.2,
    val_ratio=0.1,
    random_state=42
)

print("\nData split complete:")
utils.print_batch_info(X_train, y_train, "Training Set")
utils.print_batch_info(X_val, y_val, "Validation Set")
utils.print_batch_info(X_test, y_test, "Test Set")

## Summary

In this notebook, we:
1. Loaded configuration and set up the environment
2. Generated mock inputs or loaded preprocessed arrays
3. Inspected dataset structure and label balance
4. Visualized representative EGM waveforms
5. Applied filtering and normalization
6. Created train/validation/test splits for downstream experiments

Next steps:
- `04_train_cnn_stft.ipynb` - Train the CNN-STFT model
- `05_train_transformer.ipynb` - Train the Transformer model
- `06_evaluate_models.ipynb` - Evaluate trained models